Here's a Python function to apply a standard diff patch to a file, specifically designed to work with markdown files, although it should work with any text file:

Notes:
This function assumes that the diff is in a unified diff format. If you're dealing with different diff formats, you might need to adjust how you parse or apply the diff.
The function handles line endings by using lineterm='' in difflib.unified_diff, which means it can work with different types of line endings (\n, \r\n, etc.).
This approach might not handle all edge cases or complex diff scenarios as well as specialized diff tools, but it should cover basic patching needs. Remember, for more complex or critical patching scenarios, using tools like patch or libraries like diff-match-patch might be more robust.


In [1]:
import difflib


def apply_diff(original_text, patch):
  """
  Apply a diff patch to the original text.

  :param original_text: str, the original content of the file
  :param patch: str, the diff patch to apply
  :return: str, the new content after applying the patch
  """
  # Convert the original text into lines
  original_lines = original_text.splitlines(True)

  # Parse the patch
  patch_lines = patch.splitlines()

  # Use difflib to apply the patch
  patch_set = difflib.unified_diff(
    [],
    [],
    lineterm="",  # To handle different line endings
    fromfile="original",
    tofile="patched",
  )

  # Here we simulate the patched lines by reconstructing them based on the diff
  patched_lines = []
  for line in patch_lines:
    if line.startswith("+") and not line.startswith(
      "+++"
    ):
      patched_lines.append(line[1:])
    elif line.startswith("-") and not line.startswith(
      "---"
    ):
      continue
    else:
      # If it's not an addition or deletion, it might be context or part of the header,
      # which we would typically ignore unless it's part of the actual file content
      if not line.startswith(("---", "+++", "@@")):
        patched_lines.append(line)

  # Now we need to apply these changes to the original text
  result = []
  original_index = 0
  patch_index = 0

  # Loop through the original lines and apply the patch
  while original_index < len(
    original_lines
  ) or patch_index < len(patched_lines):
    if patch_index >= len(patched_lines):
      # If we've run out of patch lines, append remaining original lines
      result.append(original_lines[original_index])
      original_index += 1
    elif original_index >= len(original_lines):
      # If we've run out of original lines, append new lines from patch
      result.append(patched_lines[patch_index])
      patch_index += 1
    else:
      # Compare lines
      if (
        original_lines[original_index]
        == patched_lines[patch_index]
      ):
        result.append(original_lines[original_index])
        original_index += 1
        patch_index += 1
      else:
        # If there's a mismatch, we assume the patch line should replace the original
        result.append(patched_lines[patch_index])
        patch_index += 1

  # Join the result into a single string
  return "".join(result)

In [2]:
# Example usage:
original_content = """# Heading
This is a markdown file.
Another line.
"""
diff = """---
+++
@@ -1,3 +1,4 @@
 # Heading
+New line added
 This is a markdown file.
 Another line.
"""

# Apply the patch
new_content = apply_diff(original_content, diff)
print(new_content)

 # HeadingNew line added This is a markdown file. Another line.# Heading
This is a markdown file.
Another line.

